In [1]:
import pandas as pd
import numpy as np
import os, math
import gc
import torch
import mlflow
import mlflow.pytorch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

from data_preprocess import process_dicom_series_safe
from dataset import RSNAAneurysmDataset, collate
from model import EffnetAneurysmClassifier
from metric import AverageMeter, auc_per_label, rsna_final_score
from utils import set_seed, LABEL_COLS

In [2]:
input_dir = "/home/khor/kaggle_kcw/kaggle-RSNA-Intracranial-Aneurysm-Detection/input/"
train_df = pd.read_csv(f"{input_dir}/train.csv") 
train_localizers_df = pd.read_csv(f"{input_dir}/train_localizers.csv")

In [21]:
def find_unused_series_instance_uid_list(train_df, train_localizers_df):
    train_localizers_validate_df = train_df.melt(
        id_vars=['SeriesInstanceUID'],
        value_vars=LABEL_COLS,
        var_name='location',
        value_name='label'
    )

    train_localizers_validate_df = train_localizers_validate_df[train_localizers_validate_df['label'] == 1].copy()
    train_localizers_validate_df = train_localizers_validate_df[["SeriesInstanceUID", "location"]]

    train_localizers_validate_groupby_df = train_localizers_validate_df.groupby('SeriesInstanceUID')['location'].apply(
        lambda locs: sorted(list(locs))
    ).reset_index(name='location')

    train_localizers_groupby_df = train_localizers_df.groupby('SeriesInstanceUID')['location'].apply(
        lambda locs: sorted(list(locs))
    ).reset_index(name='location')

    unused_series_instance_uid_list = []
    for curr_train_localizers_validate_groupby_index, curr_train_localizers_validate_groupby_row in train_localizers_validate_groupby_df.iterrows():
        curr_train_localizers_validate_groupby_row_series_instance_uid = curr_train_localizers_validate_groupby_row['SeriesInstanceUID']
        curr_train_localizers_validate_groupby_row_location = curr_train_localizers_validate_groupby_row['location']

        curr_train_localizers_groupby_row = train_localizers_groupby_df[train_localizers_groupby_df['SeriesInstanceUID'] == curr_train_localizers_validate_groupby_row_series_instance_uid]
        if curr_train_localizers_groupby_row.empty:
            unused_series_instance_uid_list.append(curr_train_localizers_validate_groupby_row_series_instance_uid)
            print(f"UID {curr_train_localizers_validate_groupby_row_series_instance_uid} not found in train_localizers_groupby_df")
        else:
            curr_train_localizers_groupby_row_location = curr_train_localizers_groupby_row.iloc[0]['location']
            if not np.array_equal(
                np.sort(np.unique(curr_train_localizers_groupby_row_location)),
                np.sort(np.unique(curr_train_localizers_validate_groupby_row_location))
            ):
                print(f"  Localizers: {curr_train_localizers_groupby_row_location}")
                print(f"  Train labels: {curr_train_localizers_validate_groupby_row_location}")

    return unused_series_instance_uid_list

In [22]:
unused_series_instance_uid_list = find_unused_series_instance_uid_list(train_df, train_localizers_df)

UID 1.2.826.0.1.3680043.8.498.12937082136541515013380696257898978214 not found in train_localizers_groupby_df
UID 1.2.826.0.1.3680043.8.498.86840850085811129970747331978337342341 not found in train_localizers_groupby_df


In [27]:
train_df = train_df[~train_df["SeriesInstanceUID"].isin(unused_series_instance_uid_list)].reset_index(drop=True)    
train_df

,SeriesInstanceUID,PatientAge,PatientSex,Modality,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,64,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.2.826.0.1.3680043.8.498.10004684224894397679...,76,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1.2.826.0.1.3680043.8.498.10005158603912009425...,58,Male,CTA,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,1.2.826.0.1.3680043.8.498.10009383108068795488...,71,Male,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1.2.826.0.1.3680043.8.498.10012790035410518400...,48,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4341,1.2.826.0.1.3680043.8.498.99915610493694667606...,62,Female,MRI T1post,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4342,1.2.826.0.1.3680043.8.498.99920680741054836990...,76,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4343,1.2.826.0.1.3680043.8.498.99953513260518059135...,44,Female,CTA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4344,1.2.826.0.1.3680043.8.498.99982144859397209076...,58,Female,MRI T2,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [29]:
train_ds = RSNAAneurysmDataset(
    df=train_df,
    input_dir=input_dir,
    target_shape=(32, 384, 384),
    label_cols=LABEL_COLS
)


In [31]:
train_ds[0]

(tensor([[[0.0039, 0.0118, 0.0118,  ..., 0.0157, 0.0157, 0.0118],
          [0.0118, 0.0745, 0.0824,  ..., 0.0902, 0.0902, 0.0824],
          [0.0118, 0.0745, 0.0784,  ..., 0.0902, 0.0902, 0.0863],
          ...,
          [0.0078, 0.0588, 0.0392,  ..., 0.0627, 0.0588, 0.0549],
          [0.0118, 0.0706, 0.0667,  ..., 0.0588, 0.0431, 0.0471],
          [0.0118, 0.0784, 0.0824,  ..., 0.0627, 0.0510, 0.0588]],
 
         [[0.0039, 0.0118, 0.0118,  ..., 0.0157, 0.0118, 0.0118],
          [0.0118, 0.0784, 0.0706,  ..., 0.0824, 0.0745, 0.0706],
          [0.0118, 0.0784, 0.0784,  ..., 0.0784, 0.0745, 0.0627],
          ...,
          [0.0078, 0.0588, 0.0510,  ..., 0.0510, 0.0510, 0.0431],
          [0.0118, 0.0667, 0.0588,  ..., 0.0667, 0.0588, 0.0588],
          [0.0118, 0.0745, 0.0706,  ..., 0.0706, 0.0667, 0.0627]],
 
         [[0.0039, 0.0118, 0.0118,  ..., 0.0157, 0.0118, 0.0118],
          [0.0118, 0.0784, 0.0745,  ..., 0.0784, 0.0627, 0.0706],
          [0.0118, 0.0824, 0.0784,  ...,